# Stage 01 - Data Ingestion: Ingestion and Parsing of PDF Documents
### Lecture 12: PyPDFLoader vs PyMuPDFLoader Parsing Benchmarks

This notebook implements production PDF document ingestion:
1. Parsing multi-page PDF documents using `PyPDFLoader`.
2. High-speed, high-fidelity PDF extraction using `PyMuPDFLoader` (fitz).
3. Metadata inspection (page numbers, document source, bounding boxes).
4. Extracting and validating page boundaries and structured paragraphs.


In [1]:
# Step 1: Locating PDF Assets
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "data").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

pdf_path = repo_root / "data" / "pdf" / "sample_financial_report.pdf"
if not pdf_path.exists():
    pdf_path = repo_root / "data" / "pdf" / "sample_paper.pdf"

print(f"[CONFIG] Target PDF File: {pdf_path}")
assert pdf_path.exists(), f"Target PDF {pdf_path} must exist."


[CONFIG] Target PDF File: D:\Code\AI\rag-mastery\data\pdf\sample_financial_report.pdf


In [2]:
# Step 2: Parsing with PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader

pypdf_loader = PyPDFLoader(str(pdf_path))
pypdf_pages = pypdf_loader.load()

print(f"[INFO] PyPDFLoader extracted {len(pypdf_pages)} pages.")
for idx, page in enumerate(pypdf_pages[:2], start=1):
    print(f"  Page {idx} Content Sample ({len(page.page_content)} chars):")
    print(f"    {page.page_content[:120].strip()}...")
    print(f"    Page Metadata: {page.metadata}")

# Assertions
assert len(pypdf_pages) > 0, "PyPDFLoader must extract at least one page."
assert "page" in pypdf_pages[0].metadata, "Metadata must contain 0-indexed page number."


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_19312\406624092.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[INFO] PyPDFLoader extracted 3 pages.
  Page 1 Content Sample (272 chars):
    Company Financial Report 2024
The financial   performance for   fiscal year 2024 shows significant   growth in profitabi...
    Page Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'D:\\Code\\AI\\rag-mastery\\data\\pdf\\sample_financial_report.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}
  Page 2 Content Sample (19 chars):
    Page 2
[Blank Page]...
    Page Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'D:\\Code\\AI\\rag-mastery\\data\\pdf\\sample_financial_report.pdf', 'total_pages': 3, 'page': 1, 'page_label': '2'}


In [3]:
# Step 3: Parsing with PyMuPDFLoader (High-Speed & Layout-Preserving)
from langchain_community.document_loaders import PyMuPDFLoader

pymupdf_loader = PyMuPDFLoader(str(pdf_path))
pymupdf_pages = pymupdf_loader.load()

print(f"[INFO] PyMuPDFLoader extracted {len(pymupdf_pages)} pages.")
print(f"  PyMuPDF Page 1 Preview: {pymupdf_pages[0].page_content[:150].strip()}...")
print(f"  PyMuPDF Metadata: {pymupdf_pages[0].metadata}")

# Assertions
assert len(pymupdf_pages) == len(pypdf_pages), "Page counts should match between parsers."
assert all(len(p.page_content) > 0 for p in pymupdf_pages), "Extracted pages must contain text."
print("[SUCCESS] PDF document parsing and comparison verified.")


[INFO] PyMuPDFLoader extracted 3 pages.
  PyMuPDF Page 1 Preview: Company Financial Report 2024
The financial   performance for   fiscal year 2024 shows significant   growth in profitability.
Revenue increased by 25%...
  PyMuPDF Metadata: {'producer': '', 'creator': '', 'creationdate': '', 'source': 'D:\\Code\\AI\\rag-mastery\\data\\pdf\\sample_financial_report.pdf', 'file_path': 'D:\\Code\\AI\\rag-mastery\\data\\pdf\\sample_financial_report.pdf', 'total_pages': 3, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}
[SUCCESS] PDF document parsing and comparison verified.
